In [1]:
# ── Export all .npz files to CSV ──────────────────────────────────────────────
# Handles three value types:
#   numeric scalars  → 0-dimensional numeric arrays
#   string scalars   → 0-dimensional string/object arrays
#   1-D arrays       → map results (400 spectra) or profile data
#   2-D+ arrays      → flattened to 1-D before export
#
# Each .npz gets:
#   {name}_scalars.csv  — one row, all scalar values (numeric + string)
#   {name}_arrays.csv   — one column per array (if arrays exist)

import numpy as np
import pandas as pd
import os

def npz_to_csv(npz_path, out_prefix=None):
    """
    Load a .npz file and export its contents to CSV.

    Parameters
    ----------
    npz_path   : str, path to the .npz file
    out_prefix : str, prefix for output CSV filenames.
                 Defaults to npz_path without extension.

    Returns
    -------
    dict with keys 'scalars' and 'arrays' containing the DataFrames.
    """
    if not os.path.exists(npz_path):
        print(f"  ✗ {npz_path} not found — skipping")
        return None

    if out_prefix is None:
        out_prefix = os.path.splitext(npz_path)[0]

    data  = np.load(npz_path, allow_pickle=True)
    store = dict(data)

    # ── Categorise ────────────────────────────────────────────────────────────
    numeric_scalars = {}
    string_scalars  = {}
    arrays          = {}

    for k, v in store.items():
        if np.ndim(v) == 0:
            # Extract Python scalar safely with .item()
            val = v.item()
            if isinstance(val, str):
                string_scalars[k] = val
            elif isinstance(val, bool):
                numeric_scalars[k] = int(val)   # bool → 0/1 for CSV clarity
            else:
                try:
                    numeric_scalars[k] = float(val)
                except (TypeError, ValueError):
                    string_scalars[k] = str(val)
        elif np.ndim(v) >= 1:
            # Flatten higher-dimensional arrays (e.g. 2D maps) to 1D
            flat = v.flatten()
            arrays[k] = flat

    print(f"\n── {npz_path} ────────────────────────────────────────────────")
    print(f"  Numeric scalars : {len(numeric_scalars)}")
    print(f"  String scalars  : {len(string_scalars)}")
    print(f"  Arrays          : {len(arrays)}")

    # ── Print string values ───────────────────────────────────────────────────
    if string_scalars:
        print(f"\n  String values:")
        for k, v in sorted(string_scalars.items()):
            print(f"    {k:<40} : {v}")

    # ── Scalars CSV ───────────────────────────────────────────────────────────
    scalars_combined = {**{k: numeric_scalars[k] for k in sorted(numeric_scalars)},
                        **{k: string_scalars[k]  for k in sorted(string_scalars)}}

    df_scalars      = pd.DataFrame([scalars_combined])
    scalars_path    = f"{out_prefix}_scalars.csv"
    df_scalars.to_csv(scalars_path, index=False)
    print(f"\n  Saved: {scalars_path}  ({len(scalars_combined)} columns)")

    # ── Arrays CSV ────────────────────────────────────────────────────────────
    df_arrays = None
    if arrays:
        lengths = {k: len(v) for k, v in arrays.items()}
        unique_lengths = set(lengths.values())

        if len(unique_lengths) == 1:
            # All same length — one combined CSV
            df_arrays    = pd.DataFrame({k: arrays[k]
                                         for k in sorted(arrays)})
            arrays_path  = f"{out_prefix}_arrays.csv"
            df_arrays.to_csv(arrays_path, index=False)
            print(f"  Saved: {arrays_path}  "
                  f"({len(arrays)} columns × {list(unique_lengths)[0]} rows)")
        else:
            # Mixed lengths — group by length, one CSV per group
            print(f"  ⚠ Arrays have mixed lengths: "
                  f"{set(lengths.values())} — saving by group")
            from itertools import groupby
            by_len = {}
            for k, v in arrays.items():
                by_len.setdefault(len(v), {})[k] = v
            for length, group in sorted(by_len.items()):
                df_group   = pd.DataFrame({k: group[k]
                                           for k in sorted(group)})
                group_path = f"{out_prefix}_arrays_n{length}.csv"
                df_group.to_csv(group_path, index=False)
                print(f"  Saved: {group_path}  "
                      f"({len(group)} columns × {length} rows)")

    return {"scalars": df_scalars, "arrays": df_arrays}


# ── Run for all three .npz files ──────────────────────────────────────────────
npz_files = {
    "fit_results"          : "fit_results.npz",
    "calibration_results"  : "calibration_results.npz",
    "linescan_results"     : "linescan_results.npz",
}

all_data = {}
for name, path in npz_files.items():
    all_data[name] = npz_to_csv(path)

print("\n── Export complete ───────────────────────────────────────────────────")
for name, result in all_data.items():
    if result is not None:
        n_scalars = len(result["scalars"].columns) if result["scalars"] is not None else 0
        print(f"  {name:<25} {n_scalars} scalar columns exported")


── fit_results.npz ────────────────────────────────────────────────
  Numeric scalars : 103
  String scalars  : 9
  Arrays          : 0

  String values:
    contam_assignment                        : Residual MMA monomer / PMMA oligomer — C=CH2 bending
    d_model                                  : double
    d_shoulder_type                          : D⁻ lower shoulder (~1289 cm⁻¹) — strain/heavy defects
    g_model                                  : double
    si2to_model                              : triple
    td_fwhm_verdict                          : monolayer
    td_layer_name                            : Monolayer
    td_model                                 : double
    td_split_type                            : strain-like monolayer (split ~13 cm⁻¹)

  Saved: fit_results_scalars.csv  (112 columns)

── calibration_results.npz ────────────────────────────────────────────────
  Numeric scalars : 16
  String scalars  : 0
  Arrays          : 0

  Saved: calibration_results_scala